In [0]:
from pyspark.sql.functions import current_timestamp, col

# 1. Parâmetros do ambiente
# Adapte os nomes de catálogo, schema e caminho de aterrissagem (landing zone) 
# de acordo com o ambiente do seu projeto atual.
catalog = "visagio_atividade_v_energy_360"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

print(f"Iniciando a ingestão na camada Bronze para o schema: {bronze_schema}")
print(f"landing_path: {landing_path}")

Iniciando a ingestão na camada Bronze para o schema: visagio_atividade_v_energy_360.bronze
landing_path: /Volumes/visagio_atividade_v_energy_360/bronze/landing


In [0]:
# 2. Cria o catálogo
spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")

# Cria os schemas dentro do catálogo
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

# Opcional: Se for usar Volumes do Unity Catalog para a landing zone
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

DataFrame[]

In [0]:
# 3. Leitura dos arquivos brutos (Sem alteração de schemas ou valores)
print("Lendo os arquivos de unidades_consumidoras, leituras, medidores e geracao_distribuida...")

df_uc_raw = spark.read.csv(f"{landing_path}/unidades_consumidoras.csv", header=True, inferSchema=False)

# Lendo todos os arquivos (unificando a partição histórica) com a máscara *.csv
df_leituras_raw = spark.read.csv(f"{landing_path}/leituras/*.csv", header=True, inferSchema=False)

df_medidores_raw = spark.read.csv(f"{landing_path}/medidores.csv", header=True, inferSchema=False)
df_gd_raw = spark.read.csv(f"{landing_path}/geracao_distribuida.csv", header=True, inferSchema=False)

Lendo os arquivos de unidades_consumidoras, leituras, medidores e geracao_distribuida...


In [0]:
from pyspark.sql.functions import current_timestamp, col

# 4. Adição de metadados e gravação no formato Delta
print("Anexando metadados e salvando as tabelas Delta...")

# Tabela: bronze_unidades_consumidoras
df_uc_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{bronze_schema}.bronze_unidades_consumidoras")

# Tabela: bronze_leituras
df_leituras_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{bronze_schema}.bronze_leituras")

# Tabela: bronze_medidores
df_medidores_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{bronze_schema}.bronze_medidores")

# Tabela: bronze_geracao_distribuida
df_gd_raw \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{bronze_schema}.bronze_geracao_distribuida")

print("Fase 1: Ingestão na Camada Bronze concluída com sucesso.")

Anexando metadados e salvando as tabelas Delta...
Fase 1: Ingestão na Camada Bronze concluída com sucesso.
